In [6]:
# ---- pgvector 配置 ----
PG_HOST = "192.168.1.13"
PG_PORT = 15432
PG_DB = "vectordb"
PG_USER = "admin"
PG_PASSWORD = "xtsk@2024"
TAGGET_DIM = 1536

In [ ]:
HOST_IP = "192.168.1.13"

LLM_URL = f"http://{HOST_IP}:10000/llm/v1"
EMBEDDING_URL = f"http://{HOST_IP}:10000/embedding/v1"
RERANKER_URL = f"http://{HOST_IP}:10000/rerank/v1"

LLM_MODEL = "LLM"
EMBEDDING_MODEL = "EMBEDDING"
RERANKER_MODEL = "RERANKER"

API_KEY = "SII#gemr#2026!!"


In [7]:
from typing import Any, Dict, List, Optional

from openai import OpenAI

llm_client = OpenAI(api_key=API_KEY, base_url=LLM_URL)
embedding_client = OpenAI(api_key=API_KEY, base_url=EMBEDDING_URL)
reranker_client = OpenAI(api_key=API_KEY, base_url=RERANKER_URL)


def call_llm(
    prompt: str,
    system_prompt: str = "你是一个有帮助的助手。",
    temperature: float = 0,
    max_tokens: Optional[int] = None,
) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]
    kwargs: Dict[str, Any] = {
        "model": LLM_MODEL,
        "messages": messages,
        "temperature": temperature,
    }
    if max_tokens is not None:
        kwargs["max_tokens"] = max_tokens

    response = llm_client.chat.completions.create(**kwargs)
    return response.choices[0].message.content or ""



def get_embedding(text: str, truncate_dim: Optional[int] = TAGGET_DIM) -> List[float]:
    response = embedding_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=[text],
    )
    embedding = response.data[0].embedding
    if truncate_dim is not None:
        return embedding[:truncate_dim]
    return embedding



def rerank_documents(
    query: str,
    documents: List[str],
    top_n: Optional[int] = None,
    return_documents: bool = True,
) -> Dict[str, Any]:
    body: Dict[str, Any] = {
        "model": RERANKER_MODEL,
        "query": query,
        "documents": documents,
        "return_documents": return_documents,
    }
    if top_n is not None:
        body["top_n"] = top_n

    response = reranker_client.post("/rerank", cast_to=dict, body=body)
    return response


# 使用示例
# llm_text = call_llm("请总结一下 pgvector 的表结构")
# embedding_vec = get_embedding("失眠，痰热内扰，治以和胃化痰")
# rerank_result = rerank_documents(
#     query="失眠痰热证怎么治疗",
#     documents=["痰热扰心，和胃化痰。", "肝郁血虚，养血疏肝。"],
#     top_n=2,
# )

In [ ]:
rerank_documents(query="失眠痰热证怎么治疗",documents=["痰热扰心，和胃化痰。", "肝郁血虚，养血疏肝。"],top_n=2, )